In [1]:
#import libraries
import numpy as np
import cartopy.crs as ccrss

import matplotlib.pyplot as plt
import pickle
import xarray as xr
import pandas as pd

In [5]:
#Load in netcdf files

data_path = './../data/temporal/raw_data/'
#Manus Island (Tropical Location)
ds_manus = xr.open_dataset(data_path + 'manuscombined.nc') 
#Lamont, Ok (great plains location)
ds_plains = xr.open_dataset(data_path + 'wbpluvcombined.nc') 
ds_manus=ds_manus.rename({"org_precip_rate_mean": "precipitation"})
ds_plains=ds_plains.rename({"intensity_rt": "precipitation"})

## Coarsest resolution: 1 day

In [6]:
# Identify bad days that contain NaNs
bad_dates_manus = ds_manus["time.date"].where(ds_manus["precipitation"].isnull(), drop=True).values
bad_dates_plains = ds_plains["time.date"].where(ds_plains["precipitation"].isnull(), drop=True).values

# Remove those days before grouping
filtered_manus = ds_manus.sel(time=~ds_manus["time.date"].isin(bad_dates_manus))
filtered_plains = ds_plains.sel(time=~ds_plains["time.date"].isin(bad_dates_plains))

# Group once and collect valid groups
valid_groups_manus = [group for _, group in filtered_manus.groupby("time.date")]
valid_groups_plains = [group for _, group in filtered_plains.groupby("time.date")]

In [7]:
#Concatenate valid groups
ds_manus_cleaned = xr.concat(valid_groups_manus, dim='time')
ds_plains_cleaned = xr.concat(valid_groups_plains, dim='time')

In [8]:
#compare before and after cleaning data
print('Before clean manus: ' + str(len(ds_manus.groupby('time.date'))))
print('After clean manus: ' + str(len(ds_manus_cleaned.groupby('time.date'))))
print(' ')
print('Before clean plains: ' + str(len(ds_plains.groupby('time.date'))))
print('After clean plains: ' + str(len(ds_plains_cleaned.groupby('time.date'))))

Before clean manus: 3535
After clean manus: 3515
 
Before clean plains: 2676
After clean plains: 1768


In [13]:
def temporal_resample(ds):
    """
    Converts precipitation rate to per-minute values and resamples to
    2-min, 5-min, 10-min, 30-min, 1-hr, 3-hr, 6-hr, 12-hr, and daily mean rates.
    """
    #Setup precipitation rate data
    precip_rate = ds
    precip_rate = precip_rate.sortby('time')
    
    #Change given hourly rates to minute rates
    rate_min = precip_rate/(60)
    
    #Aggregate rates to larger time steps
    rate_2min = rate_min.resample(time = "2min").mean(dim='time')
    rate_5min = rate_min.resample(time = "5min").mean(dim='time')
    rate_10min = rate_min.resample(time = "10min").mean(dim='time')
    rate_30min = rate_min.resample(time = "30min").mean(dim='time')
    rate_1hr = rate_min.resample(time = "1h").mean(dim='time')
    rate_3hr = rate_min.resample(time = "3h").mean(dim='time')
    rate_6hr = rate_min.resample(time = "6h").mean(dim='time')
    rate_12hr = rate_min.resample(time = "12h").mean(dim='time')
    rate_day = rate_min.resample(time = "1d").mean(dim='time')
    
    return rate_min, rate_2min, rate_5min, rate_10min, rate_30min, rate_1hr, rate_3hr, rate_6hr, rate_12hr, rate_day

#Resample and return tuple of various rates
manus_min, manus_2min, manus_5min, manus_10min, manus_30min, manus_1hr, manus_3hr, manus_6hr, manus_12hr, manus_day = temporal_resample(ds_manus_cleaned.precipitation)
plains_min, plains_2min, plains_5min, plains_10min, plains_30min, plains_1hr, plains_3hr, plains_6hr, plains_12hr, plains_day = temporal_resample(ds_plains_cleaned.precipitation)

In [18]:
xr.Dataset({'precipitation': manus_min}).to_netcdf(path='./../data/temporal/cleaned_input/raindists/manus_1min.nc')
xr.Dataset({'precipitation': manus_2min}).to_netcdf(path='./../data/temporal/cleaned_input/raindists/manus_2min.nc')
xr.Dataset({'precipitation': manus_5min}).to_netcdf(path='./../data/temporal/cleaned_input/raindists/manus_5min.nc')
xr.Dataset({'precipitation': manus_10min}).to_netcdf(path='./../data/temporal/cleaned_input/raindists/manus_10min.nc')
xr.Dataset({'precipitation': manus_30min}).to_netcdf(path='./../data/temporal/cleaned_input/raindists/manus_30min.nc')
xr.Dataset({'precipitation': manus_1hr}).to_netcdf(path='./../data/temporal/cleaned_input/raindists/manus_1hr.nc')
xr.Dataset({'precipitation': manus_3hr}).to_netcdf(path='./../data/temporal/cleaned_input/raindists/manus_3hr.nc')
xr.Dataset({'precipitation': manus_6hr}).to_netcdf(path='./../data/temporal/cleaned_input/raindists/manus_6hr.nc')
xr.Dataset({'precipitation': manus_12hr}).to_netcdf(path='./../data/temporal/cleaned_input/raindists/manus_12hr.nc')
xr.Dataset({'precipitation': manus_day}).to_netcdf(path='./../data/temporal/cleaned_input/raindists/manus_day.nc')

xr.Dataset({'precipitation': plains_min}).to_netcdf(path='./../data/temporal/cleaned_input/raindists/plains_1min.nc')
xr.Dataset({'precipitation': plains_2min}).to_netcdf(path='./../data/temporal/cleaned_input/raindists/plains_2min.nc')
xr.Dataset({'precipitation': plains_5min}).to_netcdf(path='./../data/temporal/cleaned_input/raindists/plains_5min.nc')
xr.Dataset({'precipitation': plains_10min}).to_netcdf(path='./../data/temporal/cleaned_input/raindists/plains_10min.nc')
xr.Dataset({'precipitation': plains_30min}).to_netcdf(path='./../data/temporal/cleaned_input/raindists/plains_30min.nc')
xr.Dataset({'precipitation': plains_1hr}).to_netcdf(path='./../data/temporal/cleaned_input/raindists/plains_1hr.nc')
xr.Dataset({'precipitation': plains_3hr}).to_netcdf(path='./../data/temporal/cleaned_input/raindists/plains_3hr.nc')
xr.Dataset({'precipitation': plains_6hr}).to_netcdf(path='./../data/temporal/cleaned_input/raindists/plains_6hr.nc')
xr.Dataset({'precipitation': plains_12hr}).to_netcdf(path='./../data/temporal/cleaned_input/raindists/plains_12hr.nc')
xr.Dataset({'precipitation': plains_day}).to_netcdf(path='./../data/temporal/cleaned_input/raindists/plains_day.nc')